# RadHarmony: Evaluating a Foundation Model - Part 2




> **Please select `RadHarmony-(medgemma)` as the Python kernel.**

In [1]:
import os
import pandas as pd

GPU_ID = 2
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
DEVICE = "cuda"

## Visual question answering (MedGemma protocol)

The probe used only the image encoder; the full **vision-language model** can
answer questions about an image. `VQAEvaluator` runs this, and
`make_medgemma_vqa` reproduces the MedGemma paper's setup (§3.4, Table 9):

- **Model**: MedGemma-4B instruction-tuned (`google/medgemma-4b-it`).
- **Prompt**: the exact VQA-RAD prompt from Appendix A7.
- **Decoding**: greedy (temperature 0).
- **No system persona**: the paper omits it for medical variants, and we match
  (`system_message=None`).
- **Metrics**: token F1 over all questions plus accuracy on the yes/no ("closed")
  subset.

The paper uses VQA-RAD and SLAKE; we use VQA-RAD (swap the dataset for SLAKE).

In [ ]:
from radharmony.evaluator import VQAEvaluator
from radharmony.evaluator.backbones import make_medgemma_vqa
from radharmony.dataset import VQARadDataset

VQA_RAD_DIR = "/mnt/NAS4/datasets/external/VQA-RAD/VQA_RAD Image Folder"

# make_medgemma_vqa loads google/medgemma-4b-it and returns a transform (each
# image -> a PNG path the model reads) and an answerer (greedy decoding). It uses
# device_map="auto", so GPU pinning comes from the setup cell's
# CUDA_VISIBLE_DEVICES, not the `device` arg.
transform, answerer = make_medgemma_vqa(device=DEVICE)

vqa_ds = VQARadDataset(base_image_dir=VQA_RAD_DIR, transform=transform, cache_dir=None)

ev = VQAEvaluator(answerer, dataset=vqa_ds, output_dir="outputs/vqa")
df = ev.evaluate()
ev.save_results(df)  # writes results.csv + generations.csv to output_dir
df

One row per bucket, matching MedGemma's Table 9; each reports only the metrics that
make sense for it (hence the NaNs). Answers are normalized (lowercased, punctuation
and articles stripped, tokenized) then scored on token overlap. The examples below
use real VQA-RAD questions with illustrative predictions.

**`overall`: mean token F1 across every question.** Token F1 is the multiset
overlap: `P = overlap/|pred|`, `R = overlap/|ref|`, `F1 = 2PR/(P+R)`.

> Q: *"where is the cavitary lesion?"* · ref `"Right upper lobe"` →
> `[right, upper, lobe]` · pred `"left upper lobe"` → `[left, upper, lobe]`.
> Overlap = 2, so P = R = 2/3, **F1 ≈ 0.67** (right structure, wrong side).

**`closed`: accuracy on yes/no questions** plus their token F1. Accuracy grades the
first yes/no token the model emits, so a full sentence still counts.

> Q: *"Are the lungs normal appearing?"* · ref `"No"` · pred
> `"No, there are pulmonary nodules."` → first yes/no token is `no` → **correct
> (1.0)**. A pred of `"yes"` would score **0.0**.

**`open`: token F1 and token recall on the rest.** Recall is `overlap/|ref|`, the
fraction of reference tokens recovered; the two together separate completeness from
concision.

> Q: *"What abnormalities are seen within the lungs?"* · ref `"Pulmonary nodules"`
> · pred `"there are multiple pulmonary nodules"`. Overlap = 2 → **recall = 1.0**
> but **F1 ≈ 0.57** (P = 2/5, docked for the extra words).

Swap `make_chexagent_vqa` for CheXagent-2. These numbers cover the full VQA-RAD set;
the paper's contamination-free split isn't shipped, so values aren't bit-comparable.

### Seeing the answers

`ev.save_results(df)` writes `generations.csv` with every answer; here we pair a few
images with what the model said, running the same `answerer` on samples pulled
straight from the dataset (each sample's `img` is the PNG path the transform wrote).

In [ ]:
from viz_helpers import show_answers

show_answers(vqa_ds, answerer)

## Generating chest-X-ray reports (MedGemma protocol)

VQA asks a question; **report generation** asks the model to *write* the
findings. Same MedGemma-4B, different prompt. `make_medgemma_generator`
reproduces the paper's setup:

- **Task**: write the FINDINGS section for a MIMIC-CXR chest X-ray.
- **Indication-conditioned**: the exam's clinical indication is passed as
  context (fair: the reporting radiologist reads it too; it's input, not a
  scoring target).
- **Decoding**: greedy.
- **Metrics**: **RadGraph F1** (MedGemma's headline metric) plus **BLEU** for
  an n-gram-overlap point of comparison, both via RadEval. The demo scores
  findings; use `ref_section="impression"` for impression.

No training phase: the evaluator iterates the dataset once, generates, scores.

> **Environment.** RadGraph pins a different `transformers` from MedGemma. For a
> large run, generate here with `ev.generate_only("pairs.parquet")`, then score
> the parquet in a `radeval` venv (`uv pip install -e ".[radeval]"`). This demo
> scores inline. MIMIC-CXR is credentialed (PhysioNet).

> **First-run auth.** RadGraph downloads weights from `StanfordAIMI/RRG_scorers`
> on the first `ev.evaluate()`. If it 401s: (1) no HF token, run
> `.venv/bin/hf auth login` with a `read`-scoped token; (2) a pre-Xet token on
> the Xet-hosted repo, so regenerate it, or set `HF_HUB_DISABLE_XET=1` before
> launching Jupyter. Cached after the first successful download.

In [ ]:
from radharmony.evaluator import ReportGenerationEvaluator
from radharmony.evaluator.backbones import make_medgemma_generator
from radharmony.dataset import MIMICCXRDataset
from radharmony.harmonizer import MIMICCXRHarmonizer

In [8]:
from radharmony.evaluator import ReportGenerationEvaluator
from radharmony.evaluator.backbones import make_medgemma_generator
from radharmony.dataset import MIMICCXRDataset


# Same MedGemma-4B, now writing a findings narrative. Returns a transform (image
# -> PNG path) and a generator (greedy). Default prompt is the "<INDICATION>
# findings:" form; override with make_medgemma_generator(prompt="..."). Same
# device_map="auto" caveat as the VQA cell.
MIMIC_IMAGE_DIR = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/files/"
MIMIC_REPORT_CSV = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/cxr-study-list.csv.gz"

transform, generator = make_medgemma_generator(device=DEVICE)

# output_report=True makes each sample carry its reference report path.
df_harmonized = pd.read_parquet("df_mimic_harmonized.parquet")
gen_ds = MIMICCXRDataset(
    base_image_dir=MIMIC_IMAGE_DIR,
    report_csv_path=MIMIC_REPORT_CSV,
    transform=transform,
    output_report=True,
    cache_dir=None,
    harmonized_df=df_harmonized,
)

ev = ReportGenerationEvaluator(
    generator,
    dataset=gen_ds,
    ref_section="findings",  # swap to "impression" for that column
    use_indication=True,  # condition on the exam indication (input, not a target)
    metrics=[
        "bleu",
        "radgraph",
    ],  # BLEU (n-gram overlap) + RadGraph F1 (MedGemma's primary)
    max_samples=1000,  # small cap for the demo; remove for a full run
    output_dir="outputs/medgemma_report",
)
df = ev.evaluate()
ev.save_results(df)  # writes results.csv + generations.csv to output_dir
df

generating reports:   0%|          | 124/47139 [57:39<364:23:52, 27.90s/batch]


Using device: cuda:0


,fold,seed,bootstrap,label,bleu,radgraph_simple,radgraph_partial,radgraph_complete
0,-1,-1,-1,report,0.0,0.0,0.0,0.0


One row (`label == "report"`): a `bleu` column plus three RadGraph columns,
`radgraph_simple`, `radgraph_partial`, `radgraph_complete` (explained below). The two
metrics measure different things:

- **BLEU**: n-gram overlap with a brevity penalty. Surface-level and
  phrasing-sensitive: a clinically correct report worded differently scores low.
- **RadGraph F1**: extracts the clinical entities and relations from both reports
  and scores how well they match. Rewards the right *findings*, not the phrasing.

Reporting both shows the gap between lexical and clinical agreement. This is a close
replication, not a bit-exact reproduction: the demo scores findings
(`ref_section="impression"` for the other), MedGemma's exact prompt isn't published
(the recipe uses a faithful paraphrase, override with `prompt=`), and it runs over
whatever MIMIC-CXR split you pass.

### Seeing the generated reports

`ev.save_results(df)` writes `generations.csv` with reference and generated findings
side by side; here we show a few images beside both, running the same `generator` on
samples from the dataset (each `report` is the path to the reference report).

In [10]:
df_harmonized.report.value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
from viz_helpers import show_reports

show_reports(gen_ds, generator)

### How the RadGraph F1 columns are computed

RadEval returns three RadGraph F1 columns (`radgraph_simple`,
`radgraph_partial`, `radgraph_complete`): same metric, three severity levels.

**What RadGraph extracts.** A trained extractor (DyGIE++ with RadGraph-XL
weights) reads each report and returns a small graph:

- **Entities**: token spans labelled `ANAT-DP` (anatomy present), `OBS-DP`
  (observation present), `OBS-U` (uncertain), `OBS-DA` (absent).
- **Relations**: typed directed edges (`located_at`, `modify`,
  `suggestive_of`), always *source → target* (`small --modify--> pneumothorax`).

The same extractor runs on both reports, so its errors partially cancel.

**Reference.** *"Small pneumothorax in the right lung apex"* → `pneumothorax`
(OBS-DP) `located_at` `apex`; `small` (OBS-DP) `modify` `pneumothorax`; `right`
(ANAT-DP) `modify` `apex`; `apex` (ANAT-DP), no outgoing edge. (See the diagram
below.)

**Scoring.** Each report becomes a set of tuples, then `P = matched / |hyp|`,
`R = matched / |ref|`, `F1 = 2PR/(P+R)`. The columns differ only in the tuple
shape each entity contributes:

| Column | Tuple form | Semantics |
|---|---|---|
| `radgraph_simple`   | `(tokens, label)` | Entity only; relations ignored. |
| `radgraph_partial`  | `(tokens, label)`, or `(tokens, label, True)` if it has an outgoing edge | Entity + whether it links to *something*. |
| `radgraph_complete` | `(tokens, label)`, or one `(tokens, label, rel, target)` **per outgoing edge** | The full edge. |

**Example 1: "...in the LEFT apex" (wrong side).** Only `right`→`left` changes.
At every level 3 of 4 tuples match and the single token error is charged exactly
once: **simple = partial = complete = 0.75**.

**Example 2: "Pneumothorax is present" (ungrounded).** The extractor returns
one entity, `pneumothorax` (OBS-DP), no relations.

- *simple*: `hyp = {(pneumothorax, OBS-DP)}`, 1 of 4 ref tuples matches →
  P = 1.00, R = 0.25, **F1 ≈ 0.40**.
- *partial / complete*: the reference's `pneumothorax` is a 3-/4-tuple (it has an
  outgoing edge), the hypothesis's is a bare 2-tuple → no match → **F1 = 0.00**.

| Hypothesis | simple | partial | complete |
|---|---:|---:|---:|
| "...in the **left** apex" | 0.75 | 0.75 | 0.75 |
| "Pneumothorax is present" | 0.40 | **0.00** | 0.00 |

Two takeaways:

- **`simple ≥ partial ≥ complete`** always holds, since a match on a stricter tuple
  implies a match on the looser one.
- **`partial` is where grounding gets scored.** Naming findings in isolation
  (Example 2) earns `simple` credit but zero `partial` credit, the behavior
  most RRG papers want punished.

**Which to quote against MedGemma.** MedGemma cites "RadGraph F1" without a
level; the convention is **`radgraph_partial`** (Yu et al. 2023's `F1RadGraph`
default). RadEval hardcodes `model_type="radgraph-xl"`; older BERT-based
`radgraph` numbers aren't directly comparable.

![RadGraph F1 worked examples: the reference graph and the two hypothesis graphs, side by side](figs/RadGraph.png)

*Reference vs. the two hypotheses from the examples above. Both `modify` arrows
point modifier → head; red marks what differs from the reference.*